In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


In [3]:
def robust_counter_powerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M, nonneg= True)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    w = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    f_obj = 0
    constraints = [w - gamma*(np.zeros(N)+1) <= t]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append(-cp.power(z3,rav)/rav - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
    for j in range(M):
        constraints.append(cp.pos(-(1-m)*v[j]+lbda[j]) <= z[j])
    constraints.append(cp.abs(a)<=100)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [4]:
def robustcheckpoweru(a,R,r,p,m,r_f,rav):
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,q_b.value)

In [25]:
def squeeze_algo_putility(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    nonstop2 = False
    lowerobj = -np.inf
    firsttime = True
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= c:
            print('cut-stop',w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cuts added', iterations,' robust iterations' ,steps)
            return(w)
        if firsttime and rbvalue - c < 0.5:
            oldrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            sets = ranktoset(oldrank)
            nonstop2 = True
            firsttime = False
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
        iterations = iterations + 1
        while nonstop2:
            [w,lowerobj] = robust_counter_powerU (sets,p,R,r,m,r_f,c,rav)
            steps = steps + 1
            print('RC steps',steps,'lowerbound',lowerobj)
            newrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            if np.array_equal(newrank,oldrank):
                print('RC_rbvalue',robustcheckpoweru(w,R,r,p,m,r_f,rav)[0])
                break
            oldrank = newrank
            sets = ranktoset(newrank)
        if upperobj - lowerobj <= 1e-5:
            return('gap stop', w,'upperbound' , upperobj, 'lowerbound', lowerobj, 'cuts added', iterations,' robust iterations' ,steps)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        if firsttime == False:
            newrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            if np.array_equal(newrank,oldrank):
                nonstop2 = False
                pass
            else:
                sets2 = ranktoset(newrank)
                lowerobj2 = robust_counter_powerU (sets2,p,R,r,m,r_f,c,rav)[1]
                if lowerobj2 > lowerobj+0.001:
                    nonstop2 = True
                    sets = sets2
                    oldrank = newrank
                    steps = steps + 1
                    print('RC steps',steps)
        #[sets,added] = makesetflex(sets, np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav))
        print('upperbound' , upperobj, 'lowerbound', lowerobj, 'cuts added', iterations,' robust iterations' ,steps)

In [6]:
def normal_cutting_plane(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 0
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= c:
            return('cut-stop',w,'objective', upperobj, 'cuts added', iterations)
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        print('objective',upperobj,'cuts-added', iterations)

In [7]:
np.random.seed(5)

In [8]:
N=30
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.07174797 0.07208475 0.06633768]


In [28]:
rav= 1-2.2
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 212

In [27]:
-r_f**rav/rav

209.32386929246513

In [29]:
weights=squeeze_algo_putility(R,r,c,p,m,r_f,rav)

rbvalue 362.02363047260195
upperbound -206.63938254922442 lowerbound -inf cuts added 1  robust iterations 0
rbvalue 260.1915964859119
upperbound -207.604224098326 lowerbound -inf cuts added 2  robust iterations 0
rbvalue 224.74993591337946
upperbound -207.7773823139953 lowerbound -inf cuts added 3  robust iterations 0
rbvalue 212.0000444910687
RC steps 1 lowerbound -207.777405142743
RC_rbvalue 211.9999924032396
upperbound -207.77740272166812 lowerbound -207.777405142743 cuts added 4  robust iterations 1
rbvalue 212.00000040044586
RC steps 2 lowerbound -207.777405142743
RC_rbvalue 211.9999924032396


In [29]:
normal_cutting_plane(R,r,c,p,m,r_f,rav)

rbvalue 29.149437451233833
objective -19.231561538824966 cuts-added 1
rbvalue 22.29596352363781
objective -19.232023174500984 cuts-added 2
rbvalue 21.999999620436014


('cut-stop',
 array([0.0055672 , 0.00534531, 0.00642426]),
 'objective',
 -19.232023174500984,
 'cuts added',
 2)

In [23]:
def objective(w,p,R,rav,r_f):
    N = len(p)
    obj = 0
    for i in range(N):
        obj = (R.dot(w)[i]+(1-sum(w))*r_f)**rav/rav *p[i]+obj
    return(obj)

In [27]:
oldrank = np.argsort((R.dot(weights)+(1-sum(weights))*r_f)**rav/rav)
sets = ranktoset(oldrank)
robust_counter_powerU (sets,p,R,r,m,r_f,c,rav)

(array([0.00173319, 0.00157048, 0.00196935]), -1741.9760247152706)

In [28]:
x = 0.1
w_new = (1-x)*w_RC+x*w_cutting
#robustcheckpoweru(w_new,R,r,p,m,r_f,rav)

print(objective(w_new,p,R,rav,r_f))
print(objective(w_RC,p,R,rav,r_f))
print(objective(w_cutting,p,R,rav,r_f))
print(rb_RC)

-191.38083461878264
-195.0182264807712
-177.20680054584187
208.33476853350172


In [55]:
def norobust(R,r,p,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = []
    f_obj = 0
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,a.value)

norobust(R,r,p,r_f,rav)

(-177.20680109686649, array([0.00751825, 0.00766303, 0.00468661]))

In [30]:
weights

('gap stop',
 array([0.00035805, 0.00030139, 0.00039928]),
 'upperbound',
 -207.77740272166812,
 'lowerbound',
 -207.777405142743,
 'cuts added',
 5,
 ' robust iterations',
 2)